# 04 — PAM: área de cana e universo canavieiro baseline

Constrói o painel PAM (área plantada/colhida, qtd produzida, rendimento) a partir da Tabela 1612 do SIDRA, e define o **universo canavieiro baseline** que vai refinar a auditoria F3 do SEEG.

**Decisões metodológicas (§3.3 v2.2):**
- Apenas variáveis primárias (4 blocos: área plantada, área colhida, quantidade produzida, rendimento médio). Os 2 blocos de % derivados são descartados.
- Critério canavieiro baseline: `area_colhida_cana > 500 ha` em algum ano 2015-2019.
- Restrição ao Centro-Sul (6 UFs).

**Outputs em `data/interim/`:**
- `pam_cana_long.csv` — município × ano × variável (long)
- `pam_cana_wide.csv` — município × ano (wide, 4 colunas de variável)
- `pam_canavieiro_baseline.csv` — universo canavieiro baseline

**Output bonus:** atualização da auditoria F3 do SEEG (`outputs_pre/seeg_coverage_matrix.csv`) com o universo canavieiro PAM, esperando reduzir os 21.670 INESPERADO para próximo de zero.

In [1]:
# Setup portável — resolve a raiz do repositório sem depender do Google Drive.
# Para executar a partir do Drive, defina antes: os.environ["RENOVABIO_BASE_DIR"] = "<caminho>"
# Para executar a partir do Drive, defina antes de rodar esta célula:
import os
import sys
from pathlib import Path

if os.environ.get("RENOVABIO_BASE_DIR"):
    BASE_DIR = Path(os.environ["RENOVABIO_BASE_DIR"]).expanduser().resolve()
else:
    BASE_DIR = Path.cwd().resolve()
    while not (BASE_DIR / "requirements.txt").exists() and BASE_DIR != BASE_DIR.parent:
        BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [2]:
import importlib
from pipeline import config, normalize, pam
importlib.reload(config); importlib.reload(normalize); importlib.reload(pam)

from pipeline.config import PARAMS, interim, out_pre
from pipeline.pam import run_pam_pipeline, rerun_seeg_coverage_with_pam
print('✓ módulos carregados')

✓ módulos carregados


In [3]:
# Carrega crosswalk
cw = pd.read_csv(interim('crosswalk_centrosul.csv'), dtype={'geocode': str})
print(f'Crosswalk: {cw.shape}')

Crosswalk: (2363, 5)


## Roda pipeline PAM

In [4]:
result = run_pam_pipeline(cw, save=True)

→ Lendo tabela 1612 (formato SIDRA empilhado)...
  4 blocos primários: ['area_plantada_ha', 'area_colhida_ha', 'qtd_produzida_t', 'rendimento_kg_ha']
    area_plantada_ha: (5563, 16)
    area_colhida_ha: (5563, 16)
    qtd_produzida_t: (5563, 16)
    rendimento_kg_ha: (5563, 16)

→ Reshape long...
  pam_long (Brasil): (289276, 5)

→ Restrição ao Centro-Sul...
  pam_long (CS): (122720, 7)
  Municípios CS: 2360 (esperado 2.363)

→ Pivot wide...
  pam_wide: (22837, 8)

→ Construindo universo canavieiro baseline...
  Canavieiros baseline (area_colhida > 500ha em algum ano 2015-2019): 835 munis
  Por UF:
uf
GO     81
MG    124
MS     38
MT     22
PR    120
SP    450

→ Salvando...
  ✓ tudo salvo


## Inspeção do painel PAM

In [5]:
pam_wide = result['pam_wide']
print(f'Shape: {pam_wide.shape}')
print(f'\nColunas:')
for c in pam_wide.columns:
    print(f'  - {c}')
print(f'\nAmostra (top 5 munis × 2024):')
pam_wide[pam_wide['ano'] == 2024].sort_values('area_colhida_ha', ascending=False).head(10)

Shape: (22837, 8)

Colunas:
  - geocode
  - municipio
  - uf
  - ano
  - area_colhida_ha
  - area_plantada_ha
  - qtd_produzida_t
  - rendimento_kg_ha

Amostra (top 5 munis × 2024):


,geocode,municipio,uf,ano,area_colhida_ha,area_plantada_ha,qtd_produzida_t,rendimento_kg_ha
8859,3170107,Uberaba,MG,2024,120000.0,120000.0,9600000.0,80000.0
20110,5006002,Nova Alvorada do Sul,MS,2024,93895.0,93895.0,8116557.0,86443.0
12830,3531902,Morro Agudo,SP,2024,86986.0,86986.0,6974210.0,80176.0
22471,5218508,Quirinópolis,GO,2024,85878.0,85878.0,7127874.0,83000.0
9840,3505500,Barretos,SP,2024,79145.0,79145.0,6452296.0,81525.0
20213,5007208,Rio Brilhante,MS,2024,69932.0,69932.0,6194501.0,88579.0
3295,3127107,Frutal,MG,2024,68000.0,68000.0,4960000.0,72941.0
4457,3136306,João Pinheiro,MG,2024,64200.0,64200.0,3531000.0,55000.0
14150,3542206,Rancharia,SP,2024,64200.0,64200.0,4650776.0,72442.0
11233,3517406,Guaíra,SP,2024,63500.0,63500.0,5207000.0,82000.0


In [6]:
# Estatísticas das 4 variáveis primárias em 2024
print('Estatísticas 2024 das variáveis PAM:')
pam_wide[pam_wide['ano'] == 2024][[
    'area_plantada_ha', 'area_colhida_ha', 'qtd_produzida_t', 'rendimento_kg_ha'
]].describe()

Estatísticas 2024 das variáveis PAM:


,area_plantada_ha,area_colhida_ha,qtd_produzida_t,rendimento_kg_ha
count,1666.00000,1666.000000,1.666000e+03,1666.000000
mean,5408.02521,5371.715486,4.141944e+05,61315.563025
std,10952.36919,10906.645590,8.601391e+05,19769.014282
min,1.00000,1.000000,2.000000e+00,2000.000000
25%,24.00000,24.000000,1.020750e+03,44000.000000
50%,187.00000,187.000000,9.000000e+03,62201.500000
75%,6330.00000,6284.250000,4.796000e+05,79000.000000
max,120000.00000,120000.000000,9.600000e+06,133500.000000


## Universo canavieiro baseline (§3.3 v2.2)

In [7]:
canavieiros = result['canavieiro_baseline']
print(f'Total: {len(canavieiros)} municípios canavieiros (>500ha em 2015-2019)')
print(f'\nPor UF:')
print(canavieiros.groupby("uf").size().to_string())
print(f'\nTop 20 por área colhida média baseline:')
canavieiros.nlargest(20, 'area_colhida_mean_baseline')[
    ['municipio', 'uf', 'area_colhida_max_baseline', 'area_colhida_mean_baseline']
]

Total: 835 municípios canavieiros (>500ha em 2015-2019)

Por UF:
uf
GO     81
MG    124
MS     38
MT     22
PR    120
SP    450

Top 20 por área colhida média baseline:


,municipio,uf,area_colhida_max_baseline,area_colhida_mean_baseline
0,Morro Agudo,SP,99000.0,97780.0
1,Rio Brilhante,MS,98002.0,87988.0
2,Nova Alvorada do Sul,MS,94925.0,87357.0
3,Uberaba,MG,84000.0,78108.0
4,Quirinópolis,GO,74396.0,71600.6
5,Barretos,SP,67200.0,66180.0
7,Guaíra,SP,62000.0,60800.0
6,Frutal,MG,62006.0,59102.0
11,Jaboticabal,SP,57550.0,56840.0
18,Mineiros,GO,52000.0,51200.0


In [8]:
# Comparação com universos anteriores
print('Comparação dos universos canavieiros considerados:\n')
print(f'  Universo total CS:                       2.363 munis')
print(f'  Tratados-ANP (subset estrito):           194 munis')
print(f'  PAM area_colhida > 500ha (§3.3 v2.2):    {len(canavieiros)} munis  ⭐')
print(f'  SEEG queima > 0 (com ruído estrutural):  ~2.167 munis (descartado)')

# Quanto da queima total SEEG concentra nos 835 do PAM?
panel_seeg = pd.read_csv(interim('seeg_outcomes_audited.csv'), dtype={'geocode': str})
queima_2024 = panel_seeg[panel_seeg['ano'] == 2024][['geocode', 'queima']]
queima_total = queima_2024['queima'].sum()
queima_canavieiros = queima_2024[queima_2024['geocode'].isin(canavieiros['geocode'])]['queima'].sum()
pct = 100 * queima_canavieiros / queima_total if queima_total > 0 else 0
print(f'\nValidação cruzada:')
print(f'  Queima 2024 total no painel SEEG: {queima_total:,.0f} tCO₂eq')
print(f'  Concentrada nos {len(canavieiros)} canavieiros PAM: {queima_canavieiros:,.0f} tCO₂eq ({pct:.1f}%)')

Comparação dos universos canavieiros considerados:

  Universo total CS:                       2.363 munis
  Tratados-ANP (subset estrito):           194 munis
  PAM area_colhida > 500ha (§3.3 v2.2):    835 munis  ⭐
  SEEG queima > 0 (com ruído estrutural):  ~2.167 munis (descartado)

Validação cruzada:
  Queima 2024 total no painel SEEG: 68,810 tCO₂eq
  Concentrada nos 835 canavieiros PAM: 68,541 tCO₂eq (99.6%)


## Refinar auditoria F3 do SEEG com universo PAM

Reroda apenas a matriz de cobertura (não o painel SEEG inteiro). Compara com a primeira versão para ver redução do INESPERADO.

In [9]:
# Antes do refinamento, salva snapshot do coverage atual para comparação
coverage_v1 = pd.read_csv(out_pre('seeg_coverage_matrix.csv'), dtype={'geocode': str})
print('F3 ANTES do refinamento (universo ANP-tratados):')
print(coverage_v1['classificacao'].value_counts().to_string())
print()

# Roda refinamento
canavieiros_geocodes = canavieiros['geocode'].tolist()
coverage_v2 = rerun_seeg_coverage_with_pam(canavieiros_geocodes, save=True)
print('F3 DEPOIS do refinamento (universo PAM > 500ha):')
print(coverage_v2['classificacao'].value_counts().to_string())

F3 ANTES do refinamento (universo ANP-tratados):
classificacao
OK                     101544
RUIDO_ESTRUTURAL         8951
ZERO_LEGITIMO            6329
OK_ZERO_CANAVIEIRO        972
ZERO_NAO_CANAVIEIRO       354

F3 DEPOIS do refinamento (universo PAM > 500ha):
classificacao
OK                     101544
RUIDO_ESTRUTURAL         8951
ZERO_LEGITIMO            6329
OK_ZERO_CANAVIEIRO        972
ZERO_NAO_CANAVIEIRO       354


## Resumo

Se o pipeline rodou com sucesso, o que foi gerado:

**Em `data/interim/`:**
- `pam_cana_long.csv` (~120k linhas: 2360 munis × 13 anos × 4 variáveis)
- `pam_cana_wide.csv` (~22k linhas: município × ano × 4 cols)
- `pam_canavieiro_baseline.csv` (~835 munis canavieiros)

**Em `outputs_pre/`:**
- `seeg_coverage_matrix.csv` atualizado com universo PAM (substitui versão anterior)

**Próximas camadas:**
- `05_mapbiomas.ipynb` — uso do solo, refina ainda mais o universo canavieiro com `mb_share_cana > 5%`
- `06_sicar.ipynb` — outcomes H1c
- `07_psm_baseline.ipynb` — covariáveis baseline
- `09_assembly.ipynb` — painel completo + canavieiro